In [1]:
"""
Lab 1: Symbolic (rule-based) vs probability-threshold screening.
NOTE: probability values are INVENTED demonstration inputs. No model is trained,
so this is a behaviour comparison, NOT an accuracy experiment.
"""

def symbolic(score, complete):
    return "Review" if complete and score >= 70 else "Hold"

def threshold(probability):
    return "Review" if probability >= 0.70 else "Hold"

cases = [
    ("complete_high", 82, True, 0.81),
    ("complete_low", 68, True, 0.74),
    ("incomplete_high", 91, False, 0.88),
    ("boundary", 70, True, 0.70),
]

print("=== Part A: main comparison ===")
print("case | symbolic | threshold | disagreement")
for name, score, complete, probability in cases:
    a, b = symbolic(score, complete), threshold(probability)
    print(name, a, b, a != b, sep=" | ")

# Section 1.6: isolate the effect of each input on the boundary case.
print("\n=== Part B: boundary sensitivity (Section 1.6) ===")
print("variant | symbolic | threshold | disagreement")
variants = [
    ("score=69, prob=0.70", 69, True, 0.70),   # change score only
    ("score=70, prob=0.69", 70, True, 0.69),   # change probability only
]
for name, score, complete, probability in variants:
    a, b = symbolic(score, complete), threshold(probability)
    print(name, a, b, a != b, sep=" | ")

=== Part A: main comparison ===
case | symbolic | threshold | disagreement
complete_high | Review | Review | False
complete_low | Hold | Review | True
incomplete_high | Hold | Review | True
boundary | Review | Review | False

=== Part B: boundary sensitivity (Section 1.6) ===
variant | symbolic | threshold | disagreement
score=69, prob=0.70 | Hold | Review | True
score=70, prob=0.69 | Review | Hold | True


In [2]:
"""Lab 1, Task 2: Extended transparent rule-based scholarship checker.
Run: python lab1_rules.py

Rules (ALL must hold for "Review"; Review = flag for a human, NOT an award):
  R1 score >= 70                    (original rule)
  R2 documents_complete is True     (original rule)
  R3 attendance_percent >= 75       (NEW, named: attendance_ok)
  R4 no_disciplinary_record is True (NEW, named: clean_record)

ASSUMPTION: every threshold (70, 75) and every test value is a SYNTHETIC lab
assumption, not a real scholarship policy.
"""

SCORE_MIN = 70
ATTENDANCE_MIN = 75

def evaluate(score, documents_complete, attendance_percent, no_disciplinary_record):
    """Return (decision, reason). Every failed rule is listed explicitly."""
    failed = []
    if not score >= SCORE_MIN:
        failed.append(f"score {score} < {SCORE_MIN}")
    if not documents_complete:
        failed.append("documents incomplete")
    if not attendance_percent >= ATTENDANCE_MIN:
        failed.append(f"attendance {attendance_percent}% < {ATTENDANCE_MIN}%")
    if not no_disciplinary_record:
        failed.append("disciplinary record present")
    if failed:
        return "Hold", "; ".join(failed)
    return "Review", "all four rules satisfied"

# Exactly four core tests: (name, inputs, expected, why this case was chosen)
tests = [
    ("T1_pass",
     dict(score=85, documents_complete=True, attendance_percent=90, no_disciplinary_record=True),
     "Review", "Baseline: every rule clearly satisfied."),
    ("T2_fail_attendance",
     dict(score=85, documents_complete=True, attendance_percent=60, no_disciplinary_record=True),
     "Hold", "Only the new attendance rule fails; isolates R3."),
    ("T3_fail_clean_record",
     dict(score=85, documents_complete=True, attendance_percent=90, no_disciplinary_record=False),
     "Hold", "Only the new clean-record rule fails; isolates R4."),
    ("T4_boundary",
     dict(score=70, documents_complete=True, attendance_percent=75, no_disciplinary_record=True),
     "Review", "Every threshold exactly at its limit; checks >= (not >)."),
]

print("Thresholds are synthetic lab assumptions (score>=70, attendance>=75%).\n")
passed = 0
for name, inputs, expected, why in tests:
    actual, reason = evaluate(**inputs)
    ok = actual == expected
    passed += ok
    print(f"[{name}]")
    print(f"  input    : {inputs}")
    print(f"  expected : {expected}")
    print(f"  actual   : {actual}")
    print(f"  reason   : {reason}")
    print(f"  selected : {why}")
    print(f"  result   : {'PASS' if ok else 'FAIL'}\n")
print(f"{passed}/{len(tests)} tests passed")

Thresholds are synthetic lab assumptions (score>=70, attendance>=75%).

[T1_pass]
  input    : {'score': 85, 'documents_complete': True, 'attendance_percent': 90, 'no_disciplinary_record': True}
  expected : Review
  actual   : Review
  reason   : all four rules satisfied
  selected : Baseline: every rule clearly satisfied.
  result   : PASS

[T2_fail_attendance]
  input    : {'score': 85, 'documents_complete': True, 'attendance_percent': 60, 'no_disciplinary_record': True}
  expected : Hold
  actual   : Hold
  reason   : attendance 60% < 75%
  selected : Only the new attendance rule fails; isolates R3.
  result   : PASS

[T3_fail_clean_record]
  input    : {'score': 85, 'documents_complete': True, 'attendance_percent': 90, 'no_disciplinary_record': False}
  expected : Hold
  actual   : Hold
  reason   : disciplinary record present
  selected : Only the new clean-record rule fails; isolates R4.
  result   : PASS

[T4_boundary]
  input    : {'score': 70, 'documents_complete': True, 'att